In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, recall_score
from imblearn.over_sampling import SMOTE

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
model = joblib.load('fraud_model.pkl')

week1 = pd.read_csv('week1_baseline.csv')
week2 = pd.read_csv('week2_drift.csv')
week3 = pd.read_csv('week3_drift.csv')
week4 = pd.read_csv('week4_drift.csv')

print("Model and datasets loaded successfully!")

Model and datasets loaded successfully!


In [3]:
# Define thresholds
KS_THRESHOLD = 0.1
PSI_THRESHOLD = 0.2
AUC_THRESHOLD = 0.95

print("Thresholds set:")
print(f"KS Test threshold:  {KS_THRESHOLD}")
print(f"PSI threshold:      {PSI_THRESHOLD}")
print(f"AUC-ROC threshold:  {AUC_THRESHOLD}")
print("\nIf any threshold is crossed - retraining will trigger automatically!")

Thresholds set:
KS Test threshold:  0.1
PSI threshold:      0.2
AUC-ROC threshold:  0.95

If any threshold is crossed - retraining will trigger automatically!


In [4]:
def check_and_retrain(new_data, week_name):
    print(f"\n{'='*50}")
    print(f"Checking {week_name}...")
    print(f"{'='*50}")
    
    # Check KS Test
    ks_stat, _ = stats.ks_2samp(week1['V1'], new_data['V1'])
    print(f"KS Statistic: {ks_stat:.4f} (threshold: {KS_THRESHOLD})")
    
    # Check PSI
    base_counts, bin_edges = np.histogram(week1['V1'], bins=10)
    current_counts, _ = np.histogram(new_data['V1'], bins=bin_edges)
    base_pct = base_counts / len(week1) + 1e-6
    current_pct = current_counts / len(new_data) + 1e-6
    psi = np.sum((current_pct - base_pct) * np.log(current_pct / base_pct))
    print(f"PSI Score:    {psi:.4f} (threshold: {PSI_THRESHOLD})")
    
    # Check AUC
    X = new_data.drop('Class', axis=1)
    y = new_data['Class']
    y_pred = model.predict(X)
    auc = roc_auc_score(y, y_pred)
    print(f"AUC-ROC:      {auc:.4f} (threshold: {AUC_THRESHOLD})")
    
    # Trigger retraining if any threshold crossed
    if ks_stat > KS_THRESHOLD or psi > PSI_THRESHOLD or auc < AUC_THRESHOLD:
        print(f"\n🚨 DRIFT DETECTED - Retraining triggered for {week_name}!")
        
        # Retrain model
        X_train = new_data.drop('Class', axis=1)
        y_train = new_data['Class']
        
        smote = SMOTE(random_state=42)
        X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
        
        new_model = RandomForestClassifier(n_estimators=10, random_state=42)
        new_model.fit(X_train_sm, y_train_sm)
        
        new_auc = roc_auc_score(y, new_model.predict(X))
        print(f"✅ Retraining complete!")
        print(f"Old AUC: {auc:.4f} → New AUC: {new_auc:.4f}")
        
        return new_model
    else:
        print(f"✅ No drift detected - No retraining needed!")
        return model

print("Retraining function created successfully!")

Retraining function created successfully!


In [5]:
print("Running Adaptive Drift Monitor...")
print("Checking all weeks for drift and retraining if needed...")

model_week2 = check_and_retrain(week2, "Week 2 - Slight Drift")
model_week3 = check_and_retrain(week3, "Week 3 - Moderate Drift")
model_week4 = check_and_retrain(week4, "Week 4 - Severe Drift")


Running Adaptive Drift Monitor...
Checking all weeks for drift and retraining if needed...

Checking Week 2 - Slight Drift...
KS Statistic: 0.2776 (threshold: 0.1)
PSI Score:    0.0187 (threshold: 0.2)
AUC-ROC:      0.9674 (threshold: 0.95)

🚨 DRIFT DETECTED - Retraining triggered for Week 2 - Slight Drift!
✅ Retraining complete!
Old AUC: 0.9674 → New AUC: 1.0000

Checking Week 3 - Moderate Drift...
KS Statistic: 0.5765 (threshold: 0.1)
PSI Score:    0.3518 (threshold: 0.2)
AUC-ROC:      0.9883 (threshold: 0.95)

🚨 DRIFT DETECTED - Retraining triggered for Week 3 - Moderate Drift!
✅ Retraining complete!
Old AUC: 0.9883 → New AUC: 1.0000

Checking Week 4 - Severe Drift...
KS Statistic: 0.7457 (threshold: 0.1)
PSI Score:    0.8409 (threshold: 0.2)
AUC-ROC:      0.9255 (threshold: 0.95)

🚨 DRIFT DETECTED - Retraining triggered for Week 4 - Severe Drift!
✅ Retraining complete!
Old AUC: 0.9255 → New AUC: 1.0000


In [2]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd

# Load credentials
load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

# Test connection
headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}"
}

response = requests.get(
    f"{SUPABASE_URL}/rest/v1/transactions?limit=5",
    headers=headers
)

print("✅ Status Code:", response.status_code)
print("📊 Raw Response:", response.text[:200])

# Check if response is valid
data = response.json()
print("📋 Type of response:", type(data))

if isinstance(data, list):
    df = pd.DataFrame(data)
    print("✅ Connected! Shape:", df.shape)
    print(df.head())
else:
    print("⚠️ Response:", data)

✅ Status Code: 401
📊 Raw Response: {"message":"Invalid API key","hint":"Double check your Supabase `anon` or `service_role` API key."}
📋 Type of response: <class 'dict'>
⚠️ Response: {'message': 'Invalid API key', 'hint': 'Double check your Supabase `anon` or `service_role` API key.'}


In [3]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd

# Load credentials
load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

# Test connection
headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}"
}

response = requests.get(
    f"{SUPABASE_URL}/rest/v1/transactions?limit=5",
    headers=headers
)

print("✅ Status Code:", response.status_code)

data = response.json()

if isinstance(data, list):
    df = pd.DataFrame(data)
    print("🎉 Connected to Supabase successfully!")
    print(f"📊 Shape: {df.shape}")
    print(df.head())
else:
    print("⚠️ Response:", data)

✅ Status Code: 401
⚠️ Response: {'message': 'Invalid API key', 'hint': 'Double check your Supabase `anon` or `service_role` API key.'}


In [4]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env")

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

print("URL:", url)
print("KEY first 20 chars:", key[:20] if key else "None - key not loaded!")

URL: https://yzidkjupjidzlhetmepa.supabase.co
KEY first 20 chars: https://yzidkjupjidz


In [5]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env")

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

print("URL:", url)
print("KEY first 20 chars:", key[:20] if key else "None - key not loaded!")

URL: https://yzidkjupjidzlhetmepa.supabase.co
KEY first 20 chars: https://yzidkjupjidz


In [6]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd

# Force reload .env
load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env", override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

print("KEY first 20 chars:", SUPABASE_KEY[:20])

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}"
}

response = requests.get(
    f"{SUPABASE_URL}/rest/v1/transactions?limit=5",
    headers=headers
)

print("✅ Status Code:", response.status_code)
data = response.json()

if isinstance(data, list):
    df = pd.DataFrame(data)
    print("🎉 Connected to Supabase successfully!")
    print(f"📊 Shape: {df.shape}")
    print(df.head())
else:
    print("⚠️ Response:", data)

KEY first 20 chars: eyJhbGciOiJIUzI1NiIs
✅ Status Code: 200
🎉 Connected to Supabase successfully!
📊 Shape: (0, 0)
Empty DataFrame
Columns: []
Index: []


In [7]:
# Fetch all data from Supabase
response = requests.get(
    f"{SUPABASE_URL}/rest/v1/transactions?select=*&limit=100",
    headers=headers
)

data = response.json()
df_live = pd.DataFrame(data)

print("🎉 Live data from Supabase!")
print(f"📊 Shape: {df_live.shape}")
print(f"📋 Columns: {list(df_live.columns)}")
print(df_live.head())

🎉 Live data from Supabase!
📊 Shape: (0, 0)
📋 Columns: []
Empty DataFrame
Columns: []
Index: []


In [8]:
import pandas as pd
import requests
import json

# Load your local week1 data
df_upload = pd.read_csv(r"C:\Users\HP\Desktop\fraud_drift_project\week1_baseline.csv")

# Take only first 500 rows to keep it small
df_upload = df_upload.head(500)

# Convert to list of dictionaries
records = df_upload.to_dict(orient='records')

# Upload to Supabase in batches of 100
batch_size = 100
total = len(records)

for i in range(0, total, batch_size):
    batch = records[i:i+batch_size]
    response = requests.post(
        f"{SUPABASE_URL}/rest/v1/transactions",
        headers={
            "apikey": SUPABASE_KEY,
            "Authorization": f"Bearer {SUPABASE_KEY}",
            "Content-Type": "application/json",
            "Prefer": "return=minimal"
        },
        data=json.dumps(batch)
    )
    print(f"Batch {i//batch_size + 1}: Status {response.status_code}")

print("✅ Upload complete!")

Batch 1: Status 401
Batch 2: Status 401
Batch 3: Status 401
Batch 4: Status 401
Batch 5: Status 401
✅ Upload complete!


In [9]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Force reload credentials
load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env", override=True)
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

print("KEY starts with:", SUPABASE_KEY[:10])

# Build headers fresh
headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=minimal"
}

# Load and upload data
df_upload = pd.read_csv(r"C:\Users\HP\Desktop\fraud_drift_project\week1_baseline.csv")
df_upload = df_upload.head(100)
records = df_upload.to_dict(orient='records')

response = requests.post(
    f"{SUPABASE_URL}/rest/v1/transactions",
    headers=headers,
    data=json.dumps(records)
)

print("Upload Status:", response.status_code)
print("Response:", response.text[:200])

KEY starts with: eyJhbGciOi
Upload Status: 401
Response: {"code":"42501","details":null,"hint":null,"message":"new row violates row-level security policy for table \"transactions\""}


In [10]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Force reload credentials
load_dotenv(r"C:\Users\HP\Desktop\fraud_drift_project\.env", override=True)
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

# Build headers
headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=minimal"
}

# Load and upload 100 rows
df_upload = pd.read_csv(r"C:\Users\HP\Desktop\fraud_drift_project\week1_baseline.csv")
df_upload = df_upload.head(100)
records = df_upload.to_dict(orient='records')

response = requests.post(
    f"{SUPABASE_URL}/rest/v1/transactions",
    headers=headers,
    data=json.dumps(records)
)

print("Upload Status:", response.status_code)
print("Response:", response.text[:200])

Upload Status: 201
Response: 


In [11]:
# Read live data from Supabase
response = requests.get(
    f"{SUPABASE_URL}/rest/v1/transactions?select=*&limit=10",
    headers={
        "apikey": SUPABASE_KEY,
        "Authorization": f"Bearer {SUPABASE_KEY}"
    }
)

data = response.json()
df_live = pd.DataFrame(data)

print("🎉 Live data from Supabase!")
print(f"📊 Shape: {df_live.shape}")
print(f"📋 Columns: {list(df_live.columns[:5])}")
print(df_live.head())

🎉 Live data from Supabase!
📊 Shape: (10, 31)
📋 Columns: ['Time', 'V1', 'V2', 'V3', 'V4']
  Time        V1        V2        V3        V4        V5        V6        V7  \
0  0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1  0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2  1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3  1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4  2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.